In [10]:
import os
import cv2
import numpy as np
import pandas as pd
import rasterio
from tqdm import tqdm
import ast
import torch
from segment_anything import sam_model_registry, SamPredictor

# =========== SAM SETUP ===========
CHECKPOINT_PATH = "weights/sam_vit_h_4b8939.pth"
MODEL_TYPE = "vit_h"
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
sam = sam_model_registry[MODEL_TYPE](checkpoint=CHECKPOINT_PATH).to(device=DEVICE)
sam_predictor = SamPredictor(sam)

# =========== GEO TO PIXEL HELPERS ===========
def lonlat_to_pixel(lon, lat, transform):
    row, col = rasterio.transform.rowcol(transform, lon, lat)
    return int(col), int(row)  # (x, y) in pixel space

def parse_mask_points(mask_str, transform):
    try:
        coords = ast.literal_eval(mask_str)
        pixels = [lonlat_to_pixel(lon, lat, transform) for lon, lat in coords]
        return np.array(pixels, dtype=np.int32)
    except Exception as ex:
        print(f"[Error] Mask parse failed: {ex} | {mask_str}")
        return None

# =========== MAIN PIPELINE ===========
def run_sam_geo_csv(
    csv_path,
    geotiff_dir,
    output_dir,
    tile_col="tile_name",
    point_lon_col="longitude",
    point_lat_col="latitude",
    box_y1_col="bbox_lat_y1",
    box_x1_col="bbox_lon_x1",
    box_y2_col="bbox_lat_y2",
    box_x2_col="bbox_lon_x2",
    mask_col="mask_points_gis",
    class_label_col="class",
    conf_col="confidence"
):
    os.makedirs(output_dir, exist_ok=True)
    df = pd.read_csv(csv_path)
    df[tile_col] = df[tile_col].astype(str)

    tiff_files = {os.path.basename(f): os.path.join(geotiff_dir, f)
                  for f in os.listdir(geotiff_dir) if f.lower().endswith(('.tif', '.tiff'))}

    for tiff_file, full_path in tqdm(tiff_files.items()):
        if tiff_file not in df[tile_col].values:
            print(f"[Skip] {tiff_file} not found in {tile_col} column")
            continue

        objects = df[df[tile_col] == tiff_file].to_dict(orient='records')
        if not objects:
            print(f"[Skip] No objects for {tiff_file}")
            continue

        with rasterio.open(full_path) as src:
            img = src.read([1, 2, 3])
            img = np.transpose(img, (1, 2, 0))
            transform = src.transform

        if img.dtype != np.uint8:
            img = ((img - img.min()) / (img.max() - img.min()) * 255).astype(np.uint8)
        overlay = img.copy()

        for idx, obj in enumerate(objects):
            try:
                # ----- Point (prompt) -----
                lon = float(obj[point_lon_col])
                lat = float(obj[point_lat_col])
                px, py = lonlat_to_pixel(lon, lat, transform)

                # ----- Box (prompt) -----
                box_x1 = float(obj[box_x1_col])
                box_y1 = float(obj[box_y1_col])
                box_x2 = float(obj[box_x2_col])
                box_y2 = float(obj[box_y2_col])
                x1, y1 = lonlat_to_pixel(box_x1, box_y1, transform)
                x2, y2 = lonlat_to_pixel(box_x2, box_y2, transform)
                x1, x2 = np.clip([x1, x2], 0, overlay.shape[1] - 1)
                y1, y2 = np.clip([y1, y2], 0, overlay.shape[0] - 1)
                x1, x2 = sorted([int(x1), int(x2)])
                y1, y2 = sorted([int(y1), int(y2)])

                # ----- Mask (prompt) -----
                mask_coords = parse_mask_points(obj[mask_col], transform)
                if mask_coords is None or len(mask_coords) < 3:
                    print(f"[Skip] Invalid mask for {tiff_file}, object {idx}")
                    continue
                mask_coords_rel = mask_coords - np.array([[x1, y1]])

                # ----- Crop region -----
                if x2 <= x1 or y2 <= y1:
                    print(f"[Skip] Invalid box for {tiff_file}, object {idx}")
                    continue
                roi_rgb = overlay[y1:y2, x1:x2]
                if roi_rgb.shape[0] < 2 or roi_rgb.shape[1] < 2:
                    print(f"[Skip] Crop too small for {tiff_file}, object {idx}")
                    continue

                # ----- Prepare prompts for SAM -----
                sam_predictor.set_image(roi_rgb)
                rel_px = px - x1
                rel_py = py - y1
                input_point = np.array([[rel_px, rel_py]])
                input_label = np.array([1])
                rel_box = np.array([0, 0, x2 - x1, y2 - y1])
                input_mask = np.zeros((roi_rgb.shape[0], roi_rgb.shape[1]), dtype=np.uint8)
                cv2.fillPoly(input_mask, [mask_coords_rel.astype(np.int32)], 1)

                # ----- Run SAM -----
                masks, scores, logits = sam_predictor.predict(
                    point_coords=input_point,
                    point_labels=input_label,
                    box=rel_box,
                    mask_input=input_mask[None, :, :],
                    multimask_output=True
                )
                if masks is None or len(masks) == 0:
                    print(f"[Skip] No mask from SAM for {tiff_file}, object {idx}")
                    continue
                mask = masks[np.argmax(scores)]

                # ----- Place mask in full image -----
                mask_full = np.zeros(overlay.shape[:2], dtype=np.uint8)
                mask_resized = cv2.resize(mask.astype(np.uint8), (x2 - x1, y2 - y1), interpolation=cv2.INTER_NEAREST)
                mask_full[y1:y2, x1:x2] = mask_resized

                # ----- Draw output -----
                mask_img = np.zeros_like(overlay)
                mask_img[mask_full > 0] = [0, 0, 255]
                overlay = cv2.addWeighted(overlay, 1.0, mask_img, 0.4, 0)
                cv2.rectangle(overlay, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.circle(overlay, (int(px), int(py)), 4, (0, 255, 255), -1)
                cv2.polylines(overlay, [mask_coords], isClosed=True, color=(255, 0, 0), thickness=2)  # Draw input mask (blue)

                label = f"Cls:{obj.get(class_label_col, '')} ({float(obj.get(conf_col, 0.0)):.2f})"
                font = cv2.FONT_HERSHEY_SIMPLEX
                font_scale = 1.0
                thickness = 2
                text_size = cv2.getTextSize(label, font, font_scale, thickness)[0]
                text_origin = (x1, max(y1 - 5, text_size[1]))
                cv2.putText(overlay, label, text_origin, font, font_scale, (0, 0, 0), thickness + 2)
                cv2.putText(overlay, label, text_origin, font, font_scale, (255, 255, 255), thickness)

            except Exception as ex:
                print(f"[Error] {tiff_file}, object {idx}: {ex}")
                continue

        overlay_bgr = cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR)
        out_path = os.path.join(output_dir, os.path.splitext(tiff_file)[0] + "_sam_annotated.jpg")
        cv2.imwrite(out_path, overlay_bgr, [int(cv2.IMWRITE_JPEG_QUALITY), 90])
        print(f"[Saved] {out_path}")

# ========== USAGE ==========
run_sam_geo_csv(
    csv_path="CSV_Yolo_test/final_rect.csv",
    geotiff_dir="test_sample",
    output_dir="annotated_geotiff_jpg"
)


  0%|                                                                                            | 0/10 [00:00<?, ?it/s]

[Error] m_3108502_sw_16_030_20211103.tif, object 0: The size of tensor a (64) must match the size of tensor b (104) at non-singleton dimension 3
[Error] m_3108502_sw_16_030_20211103.tif, object 1: The size of tensor a (64) must match the size of tensor b (105) at non-singleton dimension 3
[Error] m_3108502_sw_16_030_20211103.tif, object 2: The size of tensor a (64) must match the size of tensor b (104) at non-singleton dimension 3
[Error] m_3108502_sw_16_030_20211103.tif, object 3: The size of tensor a (64) must match the size of tensor b (104) at non-singleton dimension 3
[Error] m_3108502_sw_16_030_20211103.tif, object 4: The size of tensor a (64) must match the size of tensor b (104) at non-singleton dimension 3
[Error] m_3108502_sw_16_030_20211103.tif, object 5: The size of tensor a (64) must match the size of tensor b (104) at non-singleton dimension 3
[Error] m_3108502_sw_16_030_20211103.tif, object 6: The size of tensor a (64) must match the size of tensor b (23) at non-singleto

 10%|████████▎                                                                          | 1/10 [02:57<26:34, 177.13s/it]

[Saved] annotated_geotiff_jpg/m_3108502_sw_16_030_20211103_sam_annotated.jpg
[Error] m_3108501_se_16_030_20211103.tif, object 0: The size of tensor a (64) must match the size of tensor b (155) at non-singleton dimension 3
[Error] m_3108501_se_16_030_20211103.tif, object 1: The size of tensor a (64) must match the size of tensor b (130) at non-singleton dimension 3
[Error] m_3108501_se_16_030_20211103.tif, object 2: The size of tensor a (64) must match the size of tensor b (130) at non-singleton dimension 3
[Error] m_3108501_se_16_030_20211103.tif, object 3: The size of tensor a (64) must match the size of tensor b (154) at non-singleton dimension 3
[Error] m_3108501_se_16_030_20211103.tif, object 4: The size of tensor a (64) must match the size of tensor b (84) at non-singleton dimension 3
[Error] m_3108501_se_16_030_20211103.tif, object 5: The size of tensor a (64) must match the size of tensor b (21) at non-singleton dimension 3
[Error] m_3108501_se_16_030_20211103.tif, object 6: The

 10%|████████                                                                         | 1/10 [11:34<1:44:09, 694.43s/it]


KeyboardInterrupt: 

In [9]:
import os
import cv2
import numpy as np
import pandas as pd
import rasterio
from tqdm import tqdm
import ast

def lonlat_to_pixel(lon, lat, transform):
    row, col = rasterio.transform.rowcol(transform, lon, lat)
    return int(col), int(row)  # (x, y) in pixel space

def parse_mask_points(mask_str, transform):
    try:
        coords = ast.literal_eval(mask_str)
        pixels = [lonlat_to_pixel(lon, lat, transform) for lon, lat in coords]
        return np.array(pixels, dtype=np.int32)
    except Exception as ex:
        print(f"[Error] Mask parse failed: {ex} | {mask_str}")
        return None

def visualize_geo_annotations(
    csv_path,
    geotiff_dir,
    output_dir,
    tile_col="tile_name",
    point_lon_col="longitude",
    point_lat_col="latitude",
    box_y1_col="bbox_lat_y1",
    box_x1_col="bbox_lon_x1",
    box_y2_col="bbox_lat_y2",
    box_x2_col="bbox_lon_x2",
    mask_col="mask_points_gis"
):
    os.makedirs(output_dir, exist_ok=True)
    df = pd.read_csv(csv_path)
    df[tile_col] = df[tile_col].astype(str)

    tiff_files = {os.path.basename(f): os.path.join(geotiff_dir, f)
                  for f in os.listdir(geotiff_dir) if f.lower().endswith(('.tif', '.tiff'))}

    for tiff_file, full_path in tqdm(tiff_files.items()):
        if tiff_file not in df[tile_col].values:
            print(f"[Skip] {tiff_file} not found in {tile_col} column")
            continue
        objects = df[df[tile_col] == tiff_file].to_dict(orient='records')
        if not objects:
            print(f"[Skip] No objects for {tiff_file}")
            continue

        with rasterio.open(full_path) as src:
            img = src.read([1, 2, 3])
            img = np.transpose(img, (1, 2, 0))
            transform = src.transform

        if img.dtype != np.uint8:
            img = ((img - img.min()) / (img.max() - img.min()) * 255).astype(np.uint8)
        overlay = img.copy()

        for idx, obj in enumerate(objects):
            try:
                # Point
                lon = float(obj[point_lon_col])
                lat = float(obj[point_lat_col])
                px, py = lonlat_to_pixel(lon, lat, transform)
                cv2.circle(overlay, (px, py), 6, (0, 0, 255), -1)  # Red point

                # Box
                box_x1 = float(obj[box_x1_col])
                box_y1 = float(obj[box_y1_col])
                box_x2 = float(obj[box_x2_col])
                box_y2 = float(obj[box_y2_col])
                x1, y1 = lonlat_to_pixel(box_x1, box_y1, transform)
                x2, y2 = lonlat_to_pixel(box_x2, box_y2, transform)
                x1, x2 = np.clip([x1, x2], 0, overlay.shape[1] - 1)
                y1, y2 = np.clip([y1, y2], 0, overlay.shape[0] - 1)
                x1, x2 = sorted([int(x1), int(x2)])
                y1, y2 = sorted([int(y1), int(y2)])
                cv2.rectangle(overlay, (x1, y1), (x2, y2), (0, 255, 0), 2)  # Green box

                # Mask
                mask_coords = parse_mask_points(obj[mask_col], transform)
                if mask_coords is not None and len(mask_coords) > 2:
                    cv2.polylines(overlay, [mask_coords], isClosed=True, color=(255, 0, 0), thickness=2)  # Blue mask

                print(f"[{tiff_file}][{idx}] Box: ({x1},{y1})-({x2},{y2}), Point: ({px},{py}), Mask N={len(mask_coords) if mask_coords is not None else 0}")
            except Exception as ex:
                print(f"[Error] {tiff_file}, object {idx}: {ex}")
                continue

        overlay_bgr = cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR)
        out_path = os.path.join(output_dir, os.path.splitext(tiff_file)[0] + "_geo_debug.jpg")
        cv2.imwrite(out_path, overlay_bgr, [int(cv2.IMWRITE_JPEG_QUALITY), 90])
        print(f"[Saved] {out_path}")

# --- USAGE ---
visualize_geo_annotations(
    csv_path="CSV_Yolo_test/final_rect.csv",
    geotiff_dir="test_sample",
    output_dir="annotated_geotiff_jpg"
)



  0%|                                                                                            | 0/10 [00:00<?, ?it/s]

[m_3108502_sw_16_030_20211103.tif][0] Box: (1934,1133)-(2352,1251), Point: (2143,1192), Mask N=56
[m_3108502_sw_16_030_20211103.tif][1] Box: (1900,1334)-(2321,1453), Point: (2110,1393), Mask N=61
[m_3108502_sw_16_030_20211103.tif][2] Box: (1851,1636)-(2267,1757), Point: (2059,1697), Mask N=54
[m_3108502_sw_16_030_20211103.tif][3] Box: (1884,1436)-(2301,1554), Point: (2093,1495), Mask N=53
[m_3108502_sw_16_030_20211103.tif][4] Box: (1916,1233)-(2335,1350), Point: (2126,1292), Mask N=62
[m_3108502_sw_16_030_20211103.tif][5] Box: (1869,1536)-(2285,1656), Point: (2077,1596), Mask N=59
[m_3108502_sw_16_030_20211103.tif][6] Box: (3034,2866)-(3127,2999), Point: (3080,2933), Mask N=26
[m_3108502_sw_16_030_20211103.tif][7] Box: (2921,2834)-(3030,2999), Point: (2975,2916), Mask N=36
[m_3108502_sw_16_030_20211103.tif][8] Box: (3140,2896)-(3225,2999), Point: (3183,2947), Mask N=14
[m_3108502_sw_16_030_20211103.tif][9] Box: (3249,2927)-(3327,2998), Point: (3288,2963), Mask N=8
[m_3108502_sw_16_030_

 10%|████████▍                                                                           | 1/10 [00:07<01:08,  7.65s/it]

[Saved] annotated_geotiff_jpg/m_3108502_sw_16_030_20211103_geo_debug.jpg
[m_3108501_se_16_030_20211103.tif][0] Box: (7891,623)-(8514,731), Point: (8202,677), Mask N=15
[m_3108501_se_16_030_20211103.tif][1] Box: (7132,1840)-(7652,1947), Point: (7392,1894), Mask N=22
[m_3108501_se_16_030_20211103.tif][2] Box: (7123,1962)-(7646,2066), Point: (7384,2014), Mask N=34
[m_3108501_se_16_030_20211103.tif][3] Box: (7888,502)-(8507,607), Point: (8197,555), Mask N=28
[m_3108501_se_16_030_20211103.tif][4] Box: (6699,1931)-(7037,2023), Point: (6868,1977), Mask N=19
[m_3108501_se_16_030_20211103.tif][5] Box: (6902,2188)-(6986,2807), Point: (6944,2497), Mask N=11
[m_3108501_se_16_030_20211103.tif][6] Box: (7882,379)-(8504,483), Point: (8193,431), Mask N=18
[m_3108501_se_16_030_20211103.tif][7] Box: (6699,1808)-(7044,1903), Point: (6871,1855), Mask N=23
[m_3108501_se_16_030_20211103.tif][8] Box: (7663,9)-(8280,95), Point: (7971,52), Mask N=8
[m_3108501_se_16_030_20211103.tif][9] Box: (7034,2193)-(7117,2

 20%|████████████████▊                                                                   | 2/10 [00:13<00:52,  6.54s/it]

...

[Saved] annotated_geotiff_jpg/m_3108502_ne_16_030_20211103_geo_debug.jpg
[m_3108501_ne_16_030_20211103.tif][0] Box: (8095,3905)-(8718,3997), Point: (8407,3951), Mask N=14
[m_3108501_ne_16_030_20211103.tif][1] Box: (8098,3668)-(8720,3756), Point: (8409,3712), Mask N=8
[m_3108501_ne_16_030_20211103.tif][2] Box: (8097,3783)-(8715,3873), Point: (8406,3828), Mask N=16
[m_3108501_ne_16_030_20211103.tif][3] Box: (8345,18257)-(8758,18316), Point: (8551,18286), Mask N=5
[m_3108501_ne_16_030_20211103.tif][4] Box: (8341,18460)-(8754,18519), Point: (8547,18490), Mask N=6
[m_3108501_ne_16_030_20211103.tif][5] Box: (7332,18298)-(7849,18366), Point: (7590,18332), Mask N=9
[m_3108501_ne_16_030_20211103.tif][6] Box: (7341,18097)-(7854,18163), Point: (7597,18130), Mask N=4
[m_3108501_ne_16_030_20211103.tif][7] Box: (7338,18198)-(7853,18264), Point: (7596,18231), Mask N=11
[m_3108501_ne_16_030_20211103.tif][8] Box: (8341,18359)-(8755,18417), Point: (8548,18388), Mask N=4
[m_3108501_ne_16_030_20211103.tif

 90%|███████████████████████████████████████████████████████████████████████████▌        | 9/10 [00:52<00:05,  5.33s/it]

[Saved] annotated_geotiff_jpg/m_3108501_ne_16_030_20211103_geo_debug.jpg
[m_3108501_nw_16_030_20211103.tif][0] Box: (6176,7133)-(6595,7249), Point: (6386,7191), Mask N=61
[m_3108501_nw_16_030_20211103.tif][1] Box: (6127,7433)-(6546,7551), Point: (6336,7492), Mask N=59
[m_3108501_nw_16_030_20211103.tif][2] Box: (6146,7331)-(6561,7449), Point: (6353,7390), Mask N=57
[m_3108501_nw_16_030_20211103.tif][3] Box: (6161,7236)-(6579,7349), Point: (6370,7292), Mask N=57


100%|███████████████████████████████████████████████████████████████████████████████████| 10/10 [00:57<00:00,  5.76s/it]

[Saved] annotated_geotiff_jpg/m_3108501_nw_16_030_20211103_geo_debug.jpg


In [11]:
import os
import cv2
import numpy as np
import pandas as pd
import rasterio
from tqdm import tqdm
import ast
import torch
from segment_anything import sam_model_registry, SamPredictor

# =========== SAM SETUP ===========
CHECKPOINT_PATH = "weights/sam_vit_h_4b8939.pth"
MODEL_TYPE = "vit_h"
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
sam = sam_model_registry[MODEL_TYPE](checkpoint=CHECKPOINT_PATH).to(device=DEVICE)
sam_predictor = SamPredictor(sam)

def lonlat_to_pixel(lon, lat, transform):
    row, col = rasterio.transform.rowcol(transform, lon, lat)
    return int(col), int(row)  # (x, y) in pixel space

def run_sam_geo_csv(
    csv_path,
    geotiff_dir,
    output_dir,
    tile_col="tile_name",
    point_lon_col="longitude",
    point_lat_col="latitude",
    box_y1_col="bbox_lat_y1",
    box_x1_col="bbox_lon_x1",
    box_y2_col="bbox_lat_y2",
    box_x2_col="bbox_lon_x2",
    class_label_col="class",
    conf_col="confidence"
):
    os.makedirs(output_dir, exist_ok=True)
    df = pd.read_csv(csv_path)
    df[tile_col] = df[tile_col].astype(str)

    tiff_files = {os.path.basename(f): os.path.join(geotiff_dir, f)
                  for f in os.listdir(geotiff_dir) if f.lower().endswith(('.tif', '.tiff'))}

    for tiff_file, full_path in tqdm(tiff_files.items()):
        if tiff_file not in df[tile_col].values:
            print(f"[Skip] {tiff_file} not found in {tile_col} column")
            continue

        objects = df[df[tile_col] == tiff_file].to_dict(orient='records')
        if not objects:
            print(f"[Skip] No objects for {tiff_file}")
            continue

        with rasterio.open(full_path) as src:
            img = src.read([1, 2, 3])
            img = np.transpose(img, (1, 2, 0))
            transform = src.transform

        if img.dtype != np.uint8:
            img = ((img - img.min()) / (img.max() - img.min()) * 255).astype(np.uint8)
        overlay = img.copy()

        for idx, obj in enumerate(objects):
            try:
                # ----- Point (prompt) -----
                lon = float(obj[point_lon_col])
                lat = float(obj[point_lat_col])
                px, py = lonlat_to_pixel(lon, lat, transform)

                # ----- Box (prompt) -----
                box_x1 = float(obj[box_x1_col])
                box_y1 = float(obj[box_y1_col])
                box_x2 = float(obj[box_x2_col])
                box_y2 = float(obj[box_y2_col])
                x1, y1 = lonlat_to_pixel(box_x1, box_y1, transform)
                x2, y2 = lonlat_to_pixel(box_x2, box_y2, transform)
                x1, x2 = np.clip([x1, x2], 0, overlay.shape[1] - 1)
                y1, y2 = np.clip([y1, y2], 0, overlay.shape[0] - 1)
                x1, x2 = sorted([int(x1), int(x2)])
                y1, y2 = sorted([int(y1), int(y2)])

                # ----- Crop region -----
                if x2 <= x1 or y2 <= y1:
                    print(f"[Skip] Invalid box for {tiff_file}, object {idx}")
                    continue
                roi_rgb = overlay[y1:y2, x1:x2]
                if roi_rgb.shape[0] < 2 or roi_rgb.shape[1] < 2:
                    print(f"[Skip] Crop too small for {tiff_file}, object {idx}")
                    continue

                # ----- Prepare prompts for SAM -----
                sam_predictor.set_image(roi_rgb)
                rel_px = px - x1
                rel_py = py - y1
                input_point = np.array([[rel_px, rel_py]])
                input_label = np.array([1])
                rel_box = np.array([0, 0, x2 - x1, y2 - y1])

                # ----- Run SAM (no mask input) -----
                masks, scores, logits = sam_predictor.predict(
                    point_coords=input_point,
                    point_labels=input_label,
                    box=rel_box,
                    multimask_output=True
                )
                if masks is None or len(masks) == 0:
                    print(f"[Skip] No mask from SAM for {tiff_file}, object {idx}")
                    continue
                mask = masks[np.argmax(scores)]

                # ----- Place mask in full image -----
                mask_full = np.zeros(overlay.shape[:2], dtype=np.uint8)
                mask_resized = cv2.resize(mask.astype(np.uint8), (x2 - x1, y2 - y1), interpolation=cv2.INTER_NEAREST)
                mask_full[y1:y2, x1:x2] = mask_resized

                # ----- Draw output -----
                mask_img = np.zeros_like(overlay)
                mask_img[mask_full > 0] = [0, 0, 255]
                overlay = cv2.addWeighted(overlay, 1.0, mask_img, 0.4, 0)
                cv2.rectangle(overlay, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.circle(overlay, (int(px), int(py)), 4, (0, 255, 255), -1)

                label = f"Cls:{obj.get(class_label_col, '')} ({float(obj.get(conf_col, 0.0)):.2f})"
                font = cv2.FONT_HERSHEY_SIMPLEX
                font_scale = 1.0
                thickness = 2
                text_size = cv2.getTextSize(label, font, font_scale, thickness)[0]
                text_origin = (x1, max(y1 - 5, text_size[1]))
                cv2.putText(overlay, label, text_origin, font, font_scale, (0, 0, 0), thickness + 2)
                cv2.putText(overlay, label, text_origin, font, font_scale, (255, 255, 255), thickness)

            except Exception as ex:
                print(f"[Error] {tiff_file}, object {idx}: {ex}")
                continue

        overlay_bgr = cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR)
        out_path = os.path.join(output_dir, os.path.splitext(tiff_file)[0] + "_sam_annotated.jpg")
        cv2.imwrite(out_path, overlay_bgr, [int(cv2.IMWRITE_JPEG_QUALITY), 90])
        print(f"[Saved] {out_path}")

# ========== USAGE ==========
run_sam_geo_csv(
    csv_path="CSV_Yolo_test/final_rect.csv",
    geotiff_dir="test_sample",
    output_dir="annotated_geotiff_jpg"
)


 10%|████████▎                                                                          | 1/10 [03:49<34:21, 229.04s/it]

[Saved] annotated_geotiff_jpg/m_3108502_sw_16_030_20211103_sam_annotated.jpg


 20%|████████████████▏                                                                | 2/10 [15:43<1:08:37, 514.67s/it]

[Saved] annotated_geotiff_jpg/m_3108501_se_16_030_20211103_sam_annotated.jpg


 30%|████████████████████████▉                                                          | 3/10 [19:14<43:52, 376.08s/it]

...

[Saved] annotated_geotiff_jpg/m_3108502_ne_16_030_20211103_sam_annotated.jpg


 90%|██████████████████████████████████████████████████████████████████████████▋        | 9/10 [34:06<03:48, 228.99s/it]

[Saved] annotated_geotiff_jpg/m_3108501_ne_16_030_20211103_sam_annotated.jpg


100%|██████████████████████████████████████████████████████████████████████████████████| 10/10 [34:55<00:00, 209.54s/it]

[Saved] annotated_geotiff_jpg/m_3108501_nw_16_030_20211103_sam_annotated.jpg


In [12]:
import os
import cv2
import numpy as np
import pandas as pd
import rasterio
from tqdm import tqdm
import torch
from segment_anything import sam_model_registry, SamPredictor

# =========== SAM SETUP ===========
CHECKPOINT_PATH = "weights/sam_vit_h_4b8939.pth"
MODEL_TYPE = "vit_h"
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
sam = sam_model_registry[MODEL_TYPE](checkpoint=CHECKPOINT_PATH).to(device=DEVICE)
sam_predictor = SamPredictor(sam)

def lonlat_to_pixel(lon, lat, transform):
    row, col = rasterio.transform.rowcol(transform, lon, lat)
    return int(col), int(row)  # (x, y) in pixel space

def run_sam_geo_csv_batch(
    csv_path,
    geotiff_dir,
    output_dir,
    tile_col="tile_name",
    point_lon_col="longitude",
    point_lat_col="latitude",
    box_y1_col="bbox_lat_y1",
    box_x1_col="bbox_lon_x1",
    box_y2_col="bbox_lat_y2",
    box_x2_col="bbox_lon_x2",
    class_label_col="class",
    conf_col="confidence"
):
    os.makedirs(output_dir, exist_ok=True)
    df = pd.read_csv(csv_path)
    df[tile_col] = df[tile_col].astype(str)
    tiff_files = {os.path.basename(f): os.path.join(geotiff_dir, f)
                  for f in os.listdir(geotiff_dir) if f.lower().endswith(('.tif', '.tiff'))}

    for tiff_file, full_path in tqdm(tiff_files.items()):
        if tiff_file not in df[tile_col].values:
            print(f"[Skip] {tiff_file} not found in {tile_col} column")
            continue

        objects = df[df[tile_col] == tiff_file].to_dict(orient='records')
        if not objects:
            print(f"[Skip] No objects for {tiff_file}")
            continue

        with rasterio.open(full_path) as src:
            img = src.read([1, 2, 3])
            img = np.transpose(img, (1, 2, 0))
            transform = src.transform

        if img.dtype != np.uint8:
            img = ((img - img.min()) / (img.max() - img.min()) * 255).astype(np.uint8)
        overlay = img.copy()

        # Set image ONCE
        for idx, obj in enumerate(objects):
            try:
                # ----- Point (prompt) -----
                lon = float(obj[point_lon_col])
                lat = float(obj[point_lat_col])
                px, py = lonlat_to_pixel(lon, lat, transform)

                # ----- Box (prompt) -----
                box_x1 = float(obj[box_x1_col])
                box_y1 = float(obj[box_y1_col])
                box_x2 = float(obj[box_x2_col])
                box_y2 = float(obj[box_y2_col])
                x1, y1 = lonlat_to_pixel(box_x1, box_y1, transform)
                x2, y2 = lonlat_to_pixel(box_x2, box_y2, transform)
                x1, x2 = np.clip([x1, x2], 0, overlay.shape[1] - 1)
                y1, y2 = np.clip([y1, y2], 0, overlay.shape[0] - 1)
                x1, x2 = sorted([int(x1), int(x2)])
                y1, y2 = sorted([int(y1), int(y2)])

                # ----- Crop region -----
                if x2 <= x1 or y2 <= y1:
                    print(f"[Skip] Invalid box for {tiff_file}, object {idx}")
                    continue
                roi_rgb = overlay[y1:y2, x1:x2]
                if roi_rgb.shape[0] < 2 or roi_rgb.shape[1] < 2:
                    print(f"[Skip] Crop too small for {tiff_file}, object {idx}")
                    continue

                # Prompt relative to crop
                rel_px = px - x1
                rel_py = py - y1
                input_point = np.array([[rel_px, rel_py]])
                input_label = np.array([1])
                rel_box = np.array([0, 0, x2 - x1, y2 - y1])

                # Set image ONCE for each crop
                sam_predictor.set_image(roi_rgb)
                masks, scores, logits = sam_predictor.predict(
                    point_coords=input_point,
                    point_labels=input_label,
                    box=rel_box,
                    multimask_output=True
                )
                if masks is None or len(masks) == 0:
                    print(f"[Skip] No mask from SAM for {tiff_file}, object {idx}")
                    continue
                mask = masks[np.argmax(scores)]

                # Place mask in full image
                mask_full = np.zeros(overlay.shape[:2], dtype=np.uint8)
                mask_resized = cv2.resize(mask.astype(np.uint8), (x2 - x1, y2 - y1), interpolation=cv2.INTER_NEAREST)
                mask_full[y1:y2, x1:x2] = mask_resized

                # Draw
                mask_img = np.zeros_like(overlay)
                mask_img[mask_full > 0] = [0, 0, 255]
                overlay = cv2.addWeighted(overlay, 1.0, mask_img, 0.4, 0)
                cv2.rectangle(overlay, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.circle(overlay, (int(px), int(py)), 4, (0, 255, 255), -1)
                label = f"Cls:{obj.get(class_label_col, '')} ({float(obj.get(conf_col, 0.0)):.2f})"
                font = cv2.FONT_HERSHEY_SIMPLEX
                font_scale = 1.0
                thickness = 2
                text_size = cv2.getTextSize(label, font, font_scale, thickness)[0]
                text_origin = (x1, max(y1 - 5, text_size[1]))
                cv2.putText(overlay, label, text_origin, font, font_scale, (0, 0, 0), thickness + 2)
                cv2.putText(overlay, label, text_origin, font, font_scale, (255, 255, 255), thickness)

            except Exception as ex:
                print(f"[Error] {tiff_file}, object {idx}: {ex}")
                continue

        overlay_bgr = cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR)
        out_path = os.path.join(output_dir, os.path.splitext(tiff_file)[0] + "_sam_annotated.jpg")
        cv2.imwrite(out_path, overlay_bgr, [int(cv2.IMWRITE_JPEG_QUALITY), 90])
        print(f"[Saved] {out_path}")

# ========== USAGE ==========
run_sam_geo_csv_batch(
    csv_path="CSV_Yolo_test/final_rect.csv",
    geotiff_dir="test_sample",
    output_dir="annotated_geotiff_jpg"
)


 10%|████████▎                                                                          | 1/10 [03:39<32:56, 219.63s/it]

[Saved] annotated_geotiff_jpg/m_3108502_sw_16_030_20211103_sam_annotated.jpg


 20%|████████████████▏                                                                | 2/10 [15:27<1:07:34, 506.80s/it]

[Saved] annotated_geotiff_jpg/m_3108501_se_16_030_20211103_sam_annotated.jpg


 30%|████████████████████████▉                                                          | 3/10 [18:59<43:26, 372.37s/it]

...

[Saved] annotated_geotiff_jpg/m_3108502_ne_16_030_20211103_sam_annotated.jpg


 90%|██████████████████████████████████████████████████████████████████████████▋        | 9/10 [33:39<03:47, 227.60s/it]

[Saved] annotated_geotiff_jpg/m_3108501_ne_16_030_20211103_sam_annotated.jpg


100%|██████████████████████████████████████████████████████████████████████████████████| 10/10 [34:26<00:00, 206.63s/it]

[Saved] annotated_geotiff_jpg/m_3108501_nw_16_030_20211103_sam_annotated.jpg


In [14]:
import os
import cv2
import numpy as np
import pandas as pd
import rasterio
from tqdm import tqdm
import torch
from segment_anything import sam_model_registry, SamPredictor

# =========== SAM SETUP ===========
CHECKPOINT_PATH = "weights/sam_vit_h_4b8939.pth"
MODEL_TYPE = "vit_h"
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
sam = sam_model_registry[MODEL_TYPE](checkpoint=CHECKPOINT_PATH).to(device=DEVICE)
sam_predictor = SamPredictor(sam)

def lonlat_to_pixel(lon, lat, transform):
    row, col = rasterio.transform.rowcol(transform, lon, lat)
    return int(col), int(row)  # (x, y) in pixel space

def run_sam_geo_csv_batch(
    csv_path,
    geotiff_dir,
    output_dir,
    tile_col="tile_name",
    point_lon_col="longitude",
    point_lat_col="latitude",
    box_y1_col="bbox_lat_y1",
    box_x1_col="bbox_lon_x1",
    box_y2_col="bbox_lat_y2",
    box_x2_col="bbox_lon_x2",
    class_label_col="class",
    conf_col="confidence"
):
    os.makedirs(output_dir, exist_ok=True)
    df = pd.read_csv(csv_path)
    df[tile_col] = df[tile_col].astype(str)
    tiff_files = {os.path.basename(f): os.path.join(geotiff_dir, f)
                  for f in os.listdir(geotiff_dir) if f.lower().endswith(('.tif', '.tiff'))}

    for tiff_file, full_path in tqdm(tiff_files.items()):
        if tiff_file not in df[tile_col].values:
            print(f"[Skip] {tiff_file} not found in {tile_col} column")
            continue

        objects = df[df[tile_col] == tiff_file].to_dict(orient='records')
        if not objects:
            print(f"[Skip] No objects for {tiff_file}")
            continue

        with rasterio.open(full_path) as src:
            img = src.read([1, 2, 3])
            img = np.transpose(img, (1, 2, 0))
            transform = src.transform

        if img.dtype != np.uint8:
            img = ((img - img.min()) / (img.max() - img.min()) * 255).astype(np.uint8)
        overlay = img.copy()

        for idx, obj in enumerate(objects):
            try:
                # ----- Foreground Point Prompt -----
                lon = float(obj[point_lon_col])
                lat = float(obj[point_lat_col])
                px, py = lonlat_to_pixel(lon, lat, transform)

                # ----- Box Prompt -----
                box_x1 = float(obj[box_x1_col])
                box_y1 = float(obj[box_y1_col])
                box_x2 = float(obj[box_x2_col])
                box_y2 = float(obj[box_y2_col])
                x1, y1 = lonlat_to_pixel(box_x1, box_y1, transform)
                x2, y2 = lonlat_to_pixel(box_x2, box_y2, transform)
                x1, x2 = np.clip([x1, x2], 0, overlay.shape[1] - 1)
                y1, y2 = np.clip([y1, y2], 0, overlay.shape[0] - 1)
                x1, x2 = sorted([int(x1), int(x2)])
                y1, y2 = sorted([int(y1), int(y2)])

                # ----- Crop region -----
                if x2 <= x1 or y2 <= y1:
                    print(f"[Skip] Invalid box for {tiff_file}, object {idx}")
                    continue
                roi_rgb = overlay[y1:y2, x1:x2]
                if roi_rgb.shape[0] < 2 or roi_rgb.shape[1] < 2:
                    print(f"[Skip] Crop too small for {tiff_file}, object {idx}")
                    continue

                # Foreground prompt: point inside object (relative to crop)
                rel_px = px - x1
                rel_py = py - y1

                # Background prompt: top-left corner of the box (relative to crop)
                rel_x1 = 0  # x1 - x1
                rel_y1 = 0  # y1 - y1

                # Combine both prompts for SAM: [[px, py], [x1, y1]]
                input_point = np.array([
                    [rel_px, rel_py],   # foreground
                    [rel_x1, rel_y1]    # background (box corner)
                ])
                input_label = np.array([1, 0])  # 1=foreground, 0=background

                rel_box = np.array([0, 0, x2 - x1, y2 - y1])

                sam_predictor.set_image(roi_rgb)
                masks, scores, logits = sam_predictor.predict(
                    point_coords=input_point,
                    point_labels=input_label,
                    box=rel_box,
                    multimask_output=True
                )
                if masks is None or len(masks) == 0:
                    print(f"[Skip] No mask from SAM for {tiff_file}, object {idx}")
                    continue
                mask = masks[np.argmax(scores)]

                # Place mask in full image
                mask_full = np.zeros(overlay.shape[:2], dtype=np.uint8)
                mask_resized = cv2.resize(mask.astype(np.uint8), (x2 - x1, y2 - y1), interpolation=cv2.INTER_NEAREST)
                mask_full[y1:y2, x1:x2] = mask_resized

                # Draw
                mask_img = np.zeros_like(overlay)
                mask_img[mask_full > 0] = [0, 0, 255]
                overlay = cv2.addWeighted(overlay, 1.0, mask_img, 0.4, 0)
                cv2.rectangle(overlay, (x1, y1), (x2, y2), (0, 255, 0), 2)
                # Foreground point
                cv2.circle(overlay, (int(px), int(py)), 4, (0, 255, 255), -1)
                # Background point (box corner)
                cv2.circle(overlay, (x1, y1), 4, (255, 128, 0), -1)
                label = f"Cls:{obj.get(class_label_col, '')} ({float(obj.get(conf_col, 0.0)):.2f})"
                font = cv2.FONT_HERSHEY_SIMPLEX
                font_scale = 1.0
                thickness = 2
                text_size = cv2.getTextSize(label, font, font_scale, thickness)[0]
                text_origin = (x1, max(y1 - 5, text_size[1]))
                cv2.putText(overlay, label, text_origin, font, font_scale, (0, 0, 0), thickness + 2)
                cv2.putText(overlay, label, text_origin, font, font_scale, (255, 255, 255), thickness)

            except Exception as ex:
                print(f"[Error] {tiff_file}, object {idx}: {ex}")
                continue

        overlay_bgr = cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR)
        out_path = os.path.join(output_dir, os.path.splitext(tiff_file)[0] + "_sam_annotated.jpg")
        cv2.imwrite(out_path, overlay_bgr, [int(cv2.IMWRITE_JPEG_QUALITY), 90])
        print(f"[Saved] {out_path}")

# ========== USAGE ==========
run_sam_geo_csv_batch(
    csv_path="CSV_Yolo_test/final_rect.csv",
    geotiff_dir="test_sample",
    output_dir="annotated_geotiff_jpg_1"
)


 10%|████████▎                                                                          | 1/10 [04:13<37:59, 253.30s/it]

[Saved] annotated_geotiff_jpg_1/m_3108502_sw_16_030_20211103_sam_annotated.jpg


 20%|████████████████▏                                                                | 2/10 [17:32<1:16:33, 574.14s/it]

[Saved] annotated_geotiff_jpg_1/m_3108501_se_16_030_20211103_sam_annotated.jpg


 30%|████████████████████████▉                                                          | 3/10 [21:21<48:38, 416.91s/it]

...

[Saved] annotated_geotiff_jpg_1/m_3108502_ne_16_030_20211103_sam_annotated.jpg


 90%|██████████████████████████████████████████████████████████████████████████▋        | 9/10 [36:06<03:52, 232.00s/it]

[Saved] annotated_geotiff_jpg_1/m_3108501_ne_16_030_20211103_sam_annotated.jpg


100%|██████████████████████████████████████████████████████████████████████████████████| 10/10 [36:51<00:00, 221.20s/it]

[Saved] annotated_geotiff_jpg_1/m_3108501_nw_16_030_20211103_sam_annotated.jpg
